# Classical Feature and Template Matching

> **Intermediate · Classical vision**


## Why this matters

Template matching and local-feature matching both localize visual correspondences, but their assumptions differ. Learning the boundary between them prevents brittle systems.

**Where it appears:** Logo/localization tasks, panorama stitching, visual inspection, planar registration, and image retrieval prototypes.


## Learning Objectives

- Detect keypoints with ORB (a free, patent-unencumbered alternative to SIFT)
- Compute and match binary descriptors between two images
- Filter matches with Lowe's ratio test and estimate a homography with RANSAC
- Apply template matching with different similarity metrics
- Handle multiple occurrences of a template with non-max suppression
- Understand when template matching fails (scale/rotation) and what to use instead


## Prerequisites

06 Drawing and Geometric Transformations; 10 Edges, Contours, and Shape Measurement

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.matchTemplate`, `cv2.minMaxLoc`, ORB, `BFMatcher`, ratio test, `cv2.findHomography`, RANSAC

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Feature Detection and Description

Keypoint features (corners, blobs) are distinctive locations that can be
re-found across different views of the same scene. **ORB** (Oriented FAST
and Rotated BRIEF) is fast, free to use commercially (unlike patented SIFT/
SURF), and produces compact binary descriptors matched efficiently with
Hamming distance. Raw nearest-neighbor matches contain many false
positives; **Lowe's ratio test** filters ambiguous matches, and **RANSAC**
(via `cv2.findHomography`) robustly fits a geometric transform while
rejecting remaining outliers.


### Template Matching

Template matching slides a fixed template over an image and scores
similarity at each position (`cv2.matchTemplate`). It's fast and simple
but has a hard limitation: it is **not** scale- or rotation-invariant --
a template only matches at the same size/orientation it was captured at.
For that kind of invariance, feature-based matching (previous notebook)
is the right tool. Template matching shines for fixed-viewpoint tasks
like UI element detection or fixed-camera inspection.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Feature Detection and Description


### 1. Detecting ORB keypoints

Create two views of the same synthetic scene (one rotated) to have genuine correspondences to find -- not just decoration.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid

def detect_orb(gray: np.ndarray, n_features: int = 500):
    orb = cv2.ORB_create(nfeatures=n_features)
    keypoints, descriptors = orb.detectAndCompute(gray, None)
    return keypoints, descriptors

img_a = load_real_image("images/objects", "box.png")
img_b = load_real_image("images/objects", "box_in_scene.png")
gray_a = cv2.cvtColor(img_a, cv2.COLOR_BGR2GRAY)
gray_b = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)

kp_a, desc_a = detect_orb(gray_a)
kp_b, desc_b = detect_orb(gray_b)
print(f"Keypoints found -- view A: {len(kp_a)}, view B: {len(kp_b)}")

vis_a = cv2.drawKeypoints(
    gray_a,
    kp_a,
    None,
    color=(0, 255, 0),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
)
show_grid([("keypoints on original", vis_a)], cols=1, figsize_scale=8)


### 2. Matching descriptors with a ratio test

Use k-NN matching (k=2) and Lowe's ratio test to keep only unambiguous matches -- a match is kept only if the best candidate is meaningfully closer than the second-best.


In [ ]:
def match_descriptors(desc_a, desc_b, ratio: float = 0.75) -> list:
    bf = cv2.BFMatcher(cv2.NORM_HAMMING)
    raw_matches = bf.knnMatch(desc_a, desc_b, k=2)
    good = [m for m, n in raw_matches if m.distance < ratio * n.distance]
    return good


good_matches = match_descriptors(desc_a, desc_b)
print(
    f"Raw keypoints A: {len(kp_a)} -> good matches after ratio test: {len(good_matches)}"
)

match_vis = cv2.drawMatches(
    gray_a,
    kp_a,
    gray_b,
    kp_b,
    good_matches[:40],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)
show_grid([("good matches (box vs scene)", match_vis)], cols=1, figsize_scale=12)


### 3. Robust geometric verification with RANSAC

Even after the ratio test, some matches are wrong. `cv2.findHomography` with RANSAC fits the dominant transform and reports which matches are geometric inliers.


In [ ]:
def estimate_homography(kp_a, kp_b, matches, ransac_threshold: float = 5.0):
    if len(matches) < 4:
        raise ValueError("Need at least 4 matches to estimate a homography")
    pts_a = np.float32([kp_a[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts_b = np.float32([kp_b[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    H, inlier_mask = cv2.findHomography(pts_a, pts_b, cv2.RANSAC, ransac_threshold)
    return H, inlier_mask


H, inliers = estimate_homography(kp_a, kp_b, good_matches)
inlier_count = int(inliers.sum()) if inliers is not None else 0
print(f"RANSAC inliers: {inlier_count} / {len(good_matches)} matches")
print("Estimated homography:\n", np.round(H, 3))

## Part 2: Template Matching


### 1. Basic template matching

Extract a template directly from a known location in the scene, then find it again using normalized cross-correlation -- the most robust of the six available metrics.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid

scene = load_real_image("images/standard", "messi5.jpg")
template = scene[20:170, 20:190].copy()  # the red rectangle region

result = cv2.matchTemplate(scene, template, cv2.TM_CCOEFF_NORMED)
_, max_val, _, max_loc = cv2.minMaxLoc(result)
print(f"Best match score: {max_val:.3f} at location {max_loc}")

found = scene.copy()
h, w = template.shape[:2]
cv2.rectangle(found, max_loc, (max_loc[0] + w, max_loc[1] + h), (0, 0, 255), 2)
show_grid([("template", template), ("match found", found)])

### 2. Finding multiple occurrences with NMS

For scenes with repeated instances of a template, threshold the match-score map and apply non-max suppression to avoid reporting overlapping duplicate detections.


In [ ]:
def find_all_matches(
    scene: np.ndarray, template: np.ndarray, threshold: float = 0.7
) -> list:
    """Returns a list of (x, y, w, h, score) boxes, deduplicated with simple NMS."""
    result = cv2.matchTemplate(scene, template, cv2.TM_CCOEFF_NORMED)
    h, w = template.shape[:2]
    ys, xs = np.where(result >= threshold)
    boxes = [[int(x), int(y), w, h] for x, y in zip(xs, ys)]
    scores = [float(result[y, x]) for x, y in zip(xs, ys)]
    if not boxes:
        return []
    rects = [[x, y, x + w, y + h] for x, y, w, h in boxes]
    indices = cv2.dnn.NMSBoxes(boxes, scores, threshold, 0.3)
    indices = np.array(indices).flatten() if len(indices) else []
    return [(boxes[i][0], boxes[i][1], w, h, scores[i]) for i in indices]


multi_scene = np.full((300, 500, 3), 245, dtype=np.uint8)
small_marker = np.zeros((30, 30, 3), dtype=np.uint8)
cv2.circle(small_marker, (15, 15), 13, (0, 140, 255), -1)
for x, y in [(40, 40), (200, 60), (350, 200), (120, 220)]:
    multi_scene[y : y + 30, x : x + 30] = small_marker

matches = find_all_matches(multi_scene, small_marker, threshold=0.8)
print(f"Found {len(matches)} occurrences")
annotated = multi_scene.copy()
for x, y, w, h, score in matches:
    cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 0, 255), 2)
show_grid(
    [("multi-instance scene", multi_scene), ("all matches (NMS applied)", annotated)]
)

### 3. Where template matching breaks down

Demonstrate the scale-invariance failure directly: a resized template fails to match at a good score, quantifying exactly why feature-based methods exist.


In [ ]:
scaled_template = cv2.resize(template, None, fx=1.3, fy=1.3)
result_scaled = cv2.matchTemplate(scene, scaled_template, cv2.TM_CCOEFF_NORMED)
_, max_val_scaled, _, _ = cv2.minMaxLoc(result_scaled)
print(f"Match score at original scale: {max_val:.3f}")
print(f"Match score with a 30% larger template: {max_val_scaled:.3f}  (degraded)")
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(result_scaled, cmap="viridis")
ax1.set_title("Scaled Match Heatmap")
ax2.imshow(scaled_template, cmap="gray")
ax2.set_title("Scaled Template")
plt.show()

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Feature Detection and Description: Custom Panoramic Image Stitcher

Image stitching combines overlapping images into a single wider view. By detecting keypoints in overlapping regions, matching them, and estimating a Homography transformation matrix, we warp and stitch the views together.


In [ ]:
# Generate two overlapping synthetic views
base = load_real_image("images/landscapes", "mountain.jpg")
h, w = base.shape[:2]

# View A (Left portion)
view_a = base[:, : w * 2 // 3]
# View B (Right portion, shifted by 50px)
view_b = base[:, w // 3 :]

# Detect ORB keypoints and descriptors
orb = cv2.ORB_create(nfeatures=1000)
kp_a, des_a = orb.detectAndCompute(view_a, None)
kp_b, des_b = orb.detectAndCompute(view_b, None)

# Match keypoints using Brute-Force Matcher
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des_a, des_b)

# Extract matching coordinates
src_pts = np.float32([kp_a[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
dst_pts = np.float32([kp_b[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

# Compute transformation Homography matrix
H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)

# Warp View B to canvas coordinates matching View A
canvas_w = view_a.shape[1] + view_b.shape[1]
stitched = cv2.warpPerspective(view_b, H, (canvas_w, h))

# Overlay View A on top of the warped canvas
stitched[0:h, 0 : view_a.shape[1]] = view_a

print("Panoramic image stitching completed.")
show_grid([("Left View A", view_a), ("Right View B", view_b), ("Stitched Panorama", stitched)], cols=1, figsize_scale=8)

### Mini Project — Template Matching: Sub-Pixel Accuracy Template Matching

Standard template matching locates the target at integer pixel offsets. To achieve sub-pixel alignment, we fit a 2D parabolic curve around the maximum match score coordinate to interpolate sub-pixel displacements.


In [ ]:
img = cv2.cvtColor(load_real_image("images/standard", "messi5.jpg"), cv2.COLOR_BGR2GRAY)
# Extract a template to match
template = img[30:130, 30:130]

# Perform template matching
res = cv2.matchTemplate(img, template, cv2.TM_CCOEFF_NORMED)
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res)
x, y = max_loc

# Fit 1D parabola on x-axis: [x-1, x, x+1] to find subpixel shift
if 0 < x < res.shape[1] - 1:
    s0 = res[y, x - 1]
    s1 = res[y, x]
    s2 = res[y, x + 1]
    # Vertex formula calculation
    dx = 0.5 * (s0 - s2) / (s0 - 2 * s1 + s2 + 1e-5)
    subpixel_x = x + dx
else:
    subpixel_x = float(x)

print(f"Integer Match Location X: {x}")
print(f"Sub-pixel Interploated Location X: {subpixel_x:.4f}")
result = cv2.matchTemplate(img, template, cv2.TM_CCOEFF_NORMED)
_, max_val, _, max_loc = cv2.minMaxLoc(result)
h, w = template.shape
annotated = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
cv2.rectangle(annotated, max_loc, (max_loc[0] + w, max_loc[1] + h), (0, 255, 0), 2)
show_grid([("Template", template), ("Heatmap", result), ("Match", annotated)])

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Feature Detection and Description
1. Repeat the pipeline comparing ORB against `cv2.AKAZE_create()` -- compare keypoint counts and match quality.
2. Sweep Lowe's ratio threshold from 0.6 to 0.9 and plot good-match-count vs threshold.
3. Use the estimated homography to warp view B back to align with view A, and visually confirm alignment.

Use the empty cell below to work through them.


#### Solutions — Feature Detection and Description

In [ ]:
# Solution 1: Compare ORB against AKAZE keypoints
def compare_features(image: np.ndarray) -> None:
    """Compare ORB vs AKAZE keypoint descriptors."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    orb = cv2.ORB_create()
    kp_orb = orb.detect(gray, None)

    akaze = cv2.AKAZE_create()
    kp_akaze = akaze.detect(gray, None)

    print(f"ORB keypoints detected: {len(kp_orb)}")
    print(f"AKAZE keypoints detected: {len(kp_akaze)}")

In [ ]:
# Solution 2: Lowe's ratio sweep
# Explanation: Lowe's ratio test selects correct matches by checking if the best match
# is significantly closer than the second-best match. A ratio of 0.6 is highly restrictive,
# producing few matches of very high precision. A ratio of 0.9 is loose, yielding many matches
# but containing a high rate of incorrect false matches.


In [ ]:
# Solution 3: Use the estimated homography to warp view B back to align with view A
def test_homography_warp(view_a, view_b, H) -> np.ndarray:
    """Warp view_b directly onto view_a canvas using homography."""
    h, w = view_a.shape[:2]
    warped_b = cv2.warpPerspective(view_b, H, (w, h))
    return warped_b

### Exercises — Template Matching
1. Implement a simple multi-scale template match: try the template at 5 scales and keep the best score.
2. Compare `TM_CCOEFF_NORMED` against `TM_SQDIFF_NORMED` (remember: lower is better for SQDIFF) on the same scene.
3. Add rotation to `scaled_template` by 15 degrees and measure the additional score degradation.

Use the empty cell below to work through them.


#### Solutions — Template Matching

In [ ]:
# Solution 1: Simple multi-scale template match
def multiscale_template_match(
    image: np.ndarray, template: np.ndarray
) -> tuple[int, int, float]:
    """Search for a template across multiple image scales."""
    best_val = -1.0
    best_loc = (0, 0)
    best_scale = 1.0

    t_h, t_w = template.shape[:2]

    for scale in np.linspace(0.8, 1.2, 5):
        scaled_w = int(template.shape[1] * scale)
        scaled_h = int(template.shape[0] * scale)
        if scaled_h >= image.shape[0] or scaled_w >= image.shape[1]:
            continue

        scaled_temp = cv2.resize(template, (scaled_w, scaled_h))
        res = cv2.matchTemplate(image, scaled_temp, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, max_loc = cv2.minMaxLoc(res)

        if max_val > best_val:
            best_val = max_val
            best_loc = max_loc
            best_scale = scale

    return best_loc[0], best_loc[1], best_scale


img = cv2.cvtColor(load_real_image("images/standard", "messi5.jpg"), cv2.COLOR_BGR2GRAY)
template = img[30:130, 30:130]  # original face
scaled_img = cv2.resize(img, None, fx=0.5, fy=0.5)  # shrink image
x, y, scale = multiscale_template_match(scaled_img, template)
annotated = cv2.cvtColor(scaled_img, cv2.COLOR_GRAY2BGR)
h, w = int(template.shape[0] * scale), int(template.shape[1] * scale)
cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 0, 255), 2)
show_grid([("Multi-scale match on 50% shrunk image", annotated)])

In [ ]:
# Solution 2: Compare TM_CCOEFF_NORMED against TM_SQDIFF_NORMED
# Explanation: TM_CCOEFF_NORMED scales between -1.0 and 1.0 where 1.0 is a perfect match.
# TM_SQDIFF_NORMED scales between 0.0 and 1.0 where 0.0 is a perfect match. Therefore,
# when matching using SQDIFF, developers must look for the minimum location (minMaxLoc's min_loc)
# instead of the maximum.


In [ ]:
# Solution 3: Add rotation to scaled_template by 15 degrees and measure degradation
def check_rotation_degradation() -> None:
    """Measure how ORB template matching score drops under 15 degrees rotation."""
    img = cv2.cvtColor(
        load_real_image("images/standard", "messi5.jpg"), cv2.COLOR_BGR2GRAY
    )
    temp = img[30:130, 30:130]

    # Rotate template by 15 degrees
    M = cv2.getRotationMatrix2D((50, 50), 15, 1.0)
    rotated_temp = cv2.warpAffine(temp, M, (100, 100))

    res = cv2.matchTemplate(img, rotated_temp, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, _ = cv2.minMaxLoc(res)
    print("Normal match score: 1.00 | Rotated (15 deg) match score:", f"{max_val:.4f}")


# Verify matching function
img = cv2.cvtColor(load_real_image("images/standard", "messi5.jpg"), cv2.COLOR_BGR2GRAY)
temp = img[30:130, 30:130]
loc_x, loc_y, scale = multiscale_template_match(img, temp)
print(f"Multiscale match: loc=({loc_x}, {loc_y}), scale={scale:.2f}")
check_rotation_degradation()

## Summary

You can select fixed-template or keypoint matching, remove weak matches, and verify geometry before acting on a correspondence.

- **Best Practices:** Normalize or constrain search regions, use a ratio test plus geometric verification, and report failure rather than forcing a match.
- **Common Pitfalls:** Using template matching across scale/rotation changes, accepting raw descriptor matches, and interpreting a high score as a verified match.